# Pydantic Validation for AI APIs

This notebook covers advanced Pydantic validation patterns for building robust AI APIs. FastAPI uses Pydantic for automatic request/response validation, making it essential for AI service development.

## Learning Objectives
- Master Pydantic BaseModel patterns
- Implement field validation with constraints
- Create custom validators for business logic
- Handle nested models and complex data structures
- Design proper error responses
- Implement API key validation patterns

## 1. Pydantic BaseModel Patterns

### Basic Model Definition

```python
from pydantic import BaseModel, Field
from typing import Optional, List
from datetime import datetime

class AIRequest(BaseModel):
    # Required field with description
    prompt: str = Field(..., description="Input prompt for AI")
    
    # Optional field with default
    model: str = Field(default="gemini-pro", description="Model to use")
    
    # Field with constraints
    temperature: float = Field(
        default=0.7,
        ge=0.0,  # greater than or equal
        le=2.0,  # less than or equal
        description="Temperature for generation"
    )
    
    # Integer with bounds
    max_tokens: Optional[int] = Field(
        default=None,
        ge=1,
        le=8192,
        description="Maximum tokens to generate"
    )
    
    # String with length constraints
    system_instruction: Optional[str] = Field(
        default=None,
        min_length=0,
        max_length=2048,
        description="System instruction for the model"
    )

# Example usage
request = AIRequest(
    prompt="Explain quantum computing",
    temperature=0.5,
    max_tokens=1024
)
print(request.model_dump())
```

### Field Types and Constraints

```python
from pydantic import BaseModel, Field, HttpUrl, EmailStr
from typing import Literal, Annotated
from enum import Enum

class TaskPriority(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"

class TaskRequest(BaseModel):
    # String with regex pattern
    task_id: str = Field(
        ..., 
        pattern=r"^TASK-[0-9]{6}$",
        description="Task ID in format TASK-XXXXXX"
    )
    
    # Literal type (limited values)
    priority: Literal["low", "medium", "high", "critical"] = "medium"
    
    # Enum type
    status: TaskPriority = TaskPriority.MEDIUM
    
    # Numeric with multiple constraints
    confidence_score: float = Field(
        ..., 
        ge=0.0, 
        le=1.0,
        description="Confidence score between 0 and 1"
    )
    
    # List with constraints
    tags: List[str] = Field(
        default_factory=list,
        min_length=1,
        max_length=10,
        description="1-10 tags for the task"
    )
    
    # HTTP URL validation
    webhook_url: Optional[HttpUrl] = None

# Valid request
task = TaskRequest(
    task_id="TASK-001234",
    priority="high",
    confidence_score=0.95,
    tags=["ai", "nlp"]
)

# Invalid request (will raise ValidationError)
# task_invalid = TaskRequest(task_id="INVALID", confidence_score=1.5)  # Invalid ID and score
```

## 2. Custom Validators

### Field Validators

```python
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import List, Optional

class AIConfig(BaseModel):
    model: str = Field(default="gemini-pro")
    temperature: float = Field(default=0.7)
    max_tokens: int = Field(default=1024)
    stop_sequences: List[str] = Field(default_factory=list)
    
    @field_validator('model')
    @classmethod
    def validate_model(cls, v: str) -> str:
        allowed_models = ["gemini-pro", "gemini-flash", "gemini-ultra"]
        if v not in allowed_models:
            raise ValueError(f'Model must be one of: {allowed_models}')
        return v
    
    @field_validator('temperature')
    @classmethod
    def validate_temperature(cls, v: float) -> float:
        if v < 0.0 or v > 2.0:
            raise ValueError('Temperature must be between 0.0 and 2.0')
        # Warn if temperature is very high
        if v > 1.5:
            import warnings
            warnings.warn('High temperature may produce unpredictable results')
        return v
    
    @field_validator('stop_sequences')
    @classmethod
    def validate_stop_sequences(cls, v: List[str]) -> List[str]:
        if len(v) > 5:
            raise ValueError('Maximum 5 stop sequences allowed')
        # Remove empty strings
        return [seq for seq in v if seq.strip()]

# Valid config
config = AIConfig(model="gemini-pro", temperature=0.5)

# Invalid config
# config_invalid = AIConfig(model="invalid-model")  # Raises ValidationError
```

### Model Validators

```python
from pydantic import BaseModel, Field, model_validator
from typing import Optional, Literal

class GenerationRequest(BaseModel):
    mode: Literal["standard", "streaming", "batch"]
    prompt: str = Field(..., min_length=1)
    max_tokens: Optional[int] = Field(default=None, ge=1)
    batch_size: Optional[int] = Field(default=None, ge=1)
    
    @model_validator(mode='after')
    def validate_mode_specific_fields(self):
        if self.mode == "streaming" and self.max_tokens is None:
            raise ValueError('max_tokens required for streaming mode')
        
        if self.mode == "batch":
            if self.batch_size is None:
                raise ValueError('batch_size required for batch mode')
            if self.batch_size > 100:
                raise ValueError('batch_size cannot exceed 100')
        
        return self

# Valid streaming request
streaming = GenerationRequest(
    mode="streaming",
    prompt="Tell me a story",
    max_tokens=500
)

# Invalid batch request (missing batch_size)
# batch_invalid = GenerationRequest(mode="batch", prompt="Process this")
```

## 3. Nested Models

### Complex Data Structures

```python
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
from datetime import datetime

class Message(BaseModel):
    role: str = Field(..., pattern=r"^(user|assistant|system)$")
    content: str = Field(..., min_length=1, max_length=32000)
    timestamp: Optional[datetime] = None

class ToolCall(BaseModel):
    id: str = Field(..., pattern=r"^call_[a-zA-Z0-9]+$")
    function: Dict[str, Any]
    arguments: str = Field(..., description="JSON string of arguments")

class ChatCompletionRequest(BaseModel):
    messages: List[Message] = Field(..., min_length=1, max_length=100)
    model: str = Field(default="gemini-pro")
    tools: Optional[List[ToolCall]] = None
    metadata: Optional[Dict[str, str]] = Field(default_factory=dict)

class Usage(BaseModel):
    prompt_tokens: int = Field(ge=0)
    completion_tokens: int = Field(ge=0)
    total_tokens: int = Field(ge=0)

class Choice(BaseModel):
    index: int = Field(ge=0)
    message: Message
    finish_reason: str = Field(..., pattern=r"^(stop|length|tool_calls)$")

class ChatCompletionResponse(BaseModel):
    id: str = Field(..., pattern=r"^chatcmpl-[a-zA-Z0-9]+$")
    object: str = Field(default="chat.completion")
    created: datetime
    model: str
    choices: List[Choice]
    usage: Usage

# Example nested request
request = ChatCompletionRequest(
    messages=[
        {"role": "user", "content": "Hello!"},
        {"role": "assistant", "content": "Hi there!"},
        {"role": "user", "content": "How are you?"}
    ],
    model="gemini-pro"
)

print(request.model_dump_json(indent=2))
```

## 4. Error Responses

### Structured Error Models

```python
from pydantic import BaseModel, Field
from typing import List, Optional, Any
from fastapi import HTTPException, status
from fastapi.responses import JSONResponse

class ErrorDetail(BaseModel):
    loc: List[str] = Field(..., description="Location of error (e.g., ['body', 'prompt'])")
    msg: str = Field(..., description="Error message")
    type: str = Field(..., description="Error type")

class ErrorResponse(BaseModel):
    error: str = Field(..., description="Error type")
    message: str = Field(..., description="Human-readable error message")
    details: Optional[List[ErrorDetail]] = Field(default=None, description="Detailed errors")
    status_code: int = Field(..., description="HTTP status code")
    request_id: Optional[str] = Field(default=None, description="Request ID for debugging")

# Custom exception handler
class AIServiceError(HTTPException):
    def __init__(
        self,
        status_code: int,
        error_type: str,
        message: str,
        details: Optional[List[ErrorDetail]] = None,
        request_id: Optional[str] = None
    ):
        self.error_type = error_type
        self.message = message
        self.details = details
        self.request_id = request_id
        super().__init__(status_code=status_code, detail=message)

# Usage in endpoints
from fastapi import FastAPI

app = FastAPI()

@app.exception_handler(AIServiceError)
async def ai_service_error_handler(request, exc: AIServiceError):
    return JSONResponse(
        status_code=exc.status_code,
        content=ErrorResponse(
            error=exc.error_type,
            message=exc.message,
            details=exc.details,
            status_code=exc.status_code,
            request_id=exc.request_id
        ).model_dump()
    )

@app.post("/generate")
async def generate(text: str):
    if not text.strip():
        raise AIServiceError(
            status_code=status.HTTP_400_BAD_REQUEST,
            error_type="validation_error",
            message="Text cannot be empty",
            details=[ErrorDetail(
                loc=["body", "text"],
                msg="Text cannot be empty or whitespace only",
                type="value_error.empty"
            )]
        )
    return {"result": f"Generated from: {text[:50]}..."}
```

## 5. API Key Validation Patterns

### Header-Based API Key

```python
from fastapi import FastAPI, Header, HTTPException, Depends
from pydantic import BaseModel, Field
from typing import Optional
import os

app = FastAPI()

# Load API keys from environment
VALID_API_KEYS = os.getenv("API_KEYS", "").split(",")

class APIKeyHeader(BaseModel):
    x_api_key: str = Field(..., alias="X-API-Key")

async def verify_api_key(
    x_api_key: Optional[str] = Header(None, alias="X-API-Key")
):
    if not x_api_key:
        raise HTTPException(
            status_code=401,
            detail="Missing API key. Include X-API-Key header."
        )
    
    if x_api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=403,
            detail="Invalid API key"
        )
    
    return x_api_key

@app.post("/secure-chat")
async def secure_chat(
    message: str,
    api_key: str = Depends(verify_api_key)
):
    return {"response": f"Authenticated chat with key: {api_key[:8]}..."}
```

### Bearer Token Authentication

```python
from fastapi import FastAPI, Header, HTTPException, Depends
from pydantic import BaseModel, Field
from typing import Optional
import hashlib
import hmac

app = FastAPI()

class BearerToken(BaseModel):
    token: str = Field(..., min_length=10)
    expires_at: Optional[str] = None

async def verify_bearer_token(
    authorization: Optional[str] = Header(None)
):
    if not authorization:
        raise HTTPException(
            status_code=401,
            detail="Missing Authorization header"
        )
    
    if not authorization.startswith("Bearer "):
        raise HTTPException(
            status_code=401,
            detail="Invalid Authorization format. Use: Bearer <token>"
        )
    
    token = authorization[7:]  # Remove "Bearer " prefix
    
    # Validate token (simplified example)
    if len(token) < 20:
        raise HTTPException(
            status_code=401,
            detail="Invalid token format"
        )
    
    # In production, verify against database or JWT
    # decoded = jwt.decode(token, SECRET_KEY, algorithms=["HS256"])
    
    return token

@app.post("/protected-endpoint")
async def protected_endpoint(
    data: dict,
    token: str = Depends(verify_bearer_token)
):
    return {"message": "Access granted", "token_prefix": token[:8]}
```

### Rate Limiting with API Keys

```python
from fastapi import FastAPI, Header, HTTPException, Depends
from pydantic import BaseModel
from typing import Dict
import time

app = FastAPI()

# Simple in-memory rate limiter
rate_limits: Dict[str, list] = {}
RATE_LIMIT_WINDOW = 60  # seconds
MAX_REQUESTS_PER_WINDOW = 10

class RateLimitInfo(BaseModel):
    api_key: str
    request_count: int
    window_start: float
    remaining: int

async def check_rate_limit(
    x_api_key: str = Header(..., alias="X-API-Key")
):
    current_time = time.time()
    
    # Initialize or get existing rate limit info
    if x_api_key not in rate_limits:
        rate_limits[x_api_key] = []
    
    # Remove old requests outside the window
    rate_limits[x_api_key] = [
        req_time for req_time in rate_limits[x_api_key]
        if current_time - req_time < RATE_LIMIT_WINDOW
    ]
    
    # Check if rate limit exceeded
    if len(rate_limits[x_api_key]) >= MAX_REQUESTS_PER_WINDOW:
        raise HTTPException(
            status_code=429,
            detail="Rate limit exceeded. Try again later.",
            headers={"Retry-After": str(RATE_LIMIT_WINDOW)}
        )
    
    # Add current request
    rate_limits[x_api_key].append(current_time)
    
    return RateLimitInfo(
        api_key=x_api_key[:8] + "...",
        request_count=len(rate_limits[x_api_key]),
        window_start=current_time - RATE_LIMIT_WINDOW,
        remaining=MAX_REQUESTS_PER_WINDOW - len(rate_limits[x_api_key])
    )

@app.post("/rate-limited-endpoint")
async def rate_limited_endpoint(
    data: dict,
    rate_info: RateLimitInfo = Depends(check_rate_limit)
):
    return {
        "message": "Request processed",
        "rate_limit": rate_info.model_dump()
    }
```

## Complete Example: Validation Service

Save this as `validation_service.py`:

```python
from fastapi import FastAPI, HTTPException, Depends, Header
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import List, Optional, Dict, Any
from datetime import datetime
import os

app = FastAPI(title="Validation Service")

# Configuration
VALID_API_KEYS = os.getenv("API_KEYS", "test-key-123").split(",")

# Models
class AIRequest(BaseModel):
    prompt: str = Field(..., min_length=1, max_length=10000)
    model: str = Field(default="gemini-pro")
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: Optional[int] = Field(default=None, ge=1, le=8192)
    
    @field_validator('model')
    @classmethod
    def validate_model(cls, v: str) -> str:
        allowed = ["gemini-pro", "gemini-flash"]
        if v not in allowed:
            raise ValueError(f'Model must be one of: {allowed}')
        return v

class AIResponse(BaseModel):
    id: str
    text: str
    model: str
    tokens_used: int
    latency_ms: float
    created_at: datetime

# Dependencies
async def verify_api_key(
    x_api_key: Optional[str] = Header(None, alias="X-API-Key")
):
    if not x_api_key or x_api_key not in VALID_API_KEYS:
        raise HTTPException(status_code=401, detail="Invalid API key")
    return x_api_key

# Endpoints
@app.post("/generate", response_model=AIResponse)
async def generate(
    request: AIRequest,
    api_key: str = Depends(verify_api_key)
):
    # Simulate AI generation
    import uuid
    import time
    
    start_time = time.time()
    response_text = f"Generated response for: {request.prompt[:50]}..."
    latency = (time.time() - start_time) * 1000
    
    return AIResponse(
        id=str(uuid.uuid4()),
        text=response_text,
        model=request.model,
        tokens_used=len(response_text.split()),
        latency_ms=latency,
        created_at=datetime.now()
    )

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
```

## Key Takeaways

1. **Field constraints** ensure data integrity (ge, le, min_length, max_length, pattern)
2. **Custom validators** enforce business logic beyond basic type checking
3. **Nested models** handle complex request/response structures
4. **Structured errors** provide clear feedback to API consumers
5. **API key validation** protects endpoints from unauthorized access
6. **Rate limiting** prevents abuse and ensures fair usage

## Next Steps
- Proceed to `03-ai-api-service.ipynb` for Gemini integration
- See `exercises/exercise-01.md` for hands-on validation practice
- Study FastAPI's official Pydantic documentation